In [0]:
"""
O QUE:
Ler o arquivo CSV de estabelecimentos da Receita Federal com PySpark.

POR QUÊ:
Validar a leitura dos dados antes de criar a tabela Bronze
em Delta Lake, preservando os campos originais.
"""

# Caminho do arquivo no Unity Catalog Volume.
caminho_csv = "/Volumes/workspace/default/radar_b2b_raw/estabelecimentos9.csv"

# Ler os dados sem cabeçalho e sem inferir tipos.
df_bronze = (
    spark.read
    .format("csv")
    .option("header", "false")
    .option("sep", ";")
    .option("encoding", "ISO-8859-1")
    .option("inferSchema", "false")
    .load(caminho_csv)
)

# Verificar a estrutura e visualizar cinco registros.
df_bronze.printSchema()
display(df_bronze.limit(5))

print(f"Quantidade de colunas: {len(df_bronze.columns)}")


In [0]:
# O QUE: Criar o schema Bronze no Unity Catalog.
# POR QUÊ: Organizar as tabelas brutas do Radar B2B.

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")

In [0]:

# O QUE: Criar a tabela Bronze de estabelecimentos em Delta Lake.
# POR QUÊ: Preservar os dados brutos e evitar cargas duplicadas caso o notebook seja executado novamente.

tabela = "workspace.bronze.estabelecimentos"

# Verificar se a tabela já existe antes de gravar.
if not spark.catalog.tableExists(tabela):

    df_bronze.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(tabela)

    print("Tabela Bronze criada com sucesso!")

else:
    print("A tabela já existe. Nenhuma carga realizada.")


In [0]:
# O QUE: Validar a estrutura e visualizar cinco registros da tabela Bronze.
# POR QUÊ: Confirmar que os dados foram gravados corretamente antes de iniciar as transformações na Silver.

df_validacao = spark.table("workspace.bronze.estabelecimentos")

print(f"Colunas: {len(df_validacao.columns)}")
print(f"Registros: {df_validacao.count()}")
